# Documentary GPU — Colab レンダリング

**ランタイム → ランタイムのタイプを変更 → ハードウェアアクセラレータ → T4 GPU** を選択。

- 設計の根拠: `docs/research_2026.md` / `CLAUDE.md`
- 主役: **2.5Dパララックス**（Depth Anything V2 → DepthFlow）
- 役割分担: ローカル(Claude)=設計/コーディング、**Colab=GPUレンダリング**

上から順に実行する。セル2の依存導入は初回のみ数分かかる。

In [ ]:
# ═══ セル1: コード取り込み + Drive マウント ═══
# 方法A(推奨): public repo を git clone。 方法B: Drive からコードをコピー。
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/documentary_gpu'
REPO = 'https://github.com/task2000jp/documentary_gpu.git'  # public・無認証clone可

if REPO:
    if os.path.isdir(WORK):
        subprocess.run(['git', '-C', WORK, 'pull'], check=False)
    else:
        subprocess.run(['git', 'clone', REPO, WORK], check=True)
else:
    # Drive にコード一式を置いた場合のフォールバック
    SRC = '/content/drive/MyDrive/documentary_gpu_code'
    if os.path.isdir(SRC):
        subprocess.run(['rsync', '-a', '--delete', f'{SRC}/', f'{WORK}/'], check=False)
    else:
        os.makedirs(WORK, exist_ok=True)
        print('⚠️ REPO未設定 & Driveコードも無し。手動アップロードするか REPO を設定。')

print('WORK =', WORK)

In [ ]:
# ═══ セル2: 依存インストール ═══
# torch / CUDA は Colab プリインストール版を使う（上書きしない＝CUDA不整合回避）。
%cd $WORK
!apt-get -qq install -y ffmpeg fonts-ipafont fonts-noto-cjk > /dev/null

# --- Phase 1（パララックス）に必要な最小セット。これだけで試写まで動く ---
!pip install -q transformers Pillow numpy imageio imageio-ffmpeg soundfile edge-tts

# --- Phase 2+ を使うとき（FLUX画像生成 / LTX動画）だけ ↓ を実行 ---
# !pip install -q diffusers accelerate sentencepiece bitsandbytes
# 任意（重い・高品質パララックス / 日本語TTS）:
# !pip install -q depthflow
# !pip install -q style-bert-vits2

In [ ]:
# ═══ セル3: 環境チェック（GPU/ffmpeg/モデル） ═══
%cd $WORK
!python scripts/pipeline.py doctor

In [ ]:
# ═══ セル4: アセット取り込み（Drive → ローカル作業ディレクトリ） ═══
# Drive: documentary_gpu_assets/{images,bgm,scenes}/  （ID 1CzEDLu0KzLhHWrBWv2dkVa4MMKWcc0Zc）
%cd $WORK
ASSETS = '/content/drive/MyDrive/documentary_gpu_assets'
!mkdir -p assets/images assets/bgm scenes
!cp -n $ASSETS/images/* assets/images/ 2>/dev/null || echo 'images: Drive未配置'
!cp -n $ASSETS/bgm/*    assets/bgm/    2>/dev/null || echo 'bgm: Drive未配置'
# シーンJSONも Drive 経由で受け取る場合（Claudeが投下 → ここで取得）
!cp -n $ASSETS/scenes/* scenes/ 2>/dev/null || true
print('--- assets/images ---'); !ls assets/images 2>/dev/null | head
print('--- scenes ---');        !ls scenes 2>/dev/null

In [ ]:
# ═══ セル5: 単一シーン試写（パララックス確認） ═══
%cd $WORK
!python scripts/pipeline.py render-scene scenes/test.json test_luther
from IPython.display import Video
Video('build/clips/test_luther.mp4', embed=True, width=720)

In [ ]:
# ═══ セル6: 章まるごとレンダリング ═══
%cd $WORK
CHAPTER = 'test'  # scenes/<CHAPTER>.json
!python scripts/pipeline.py render $CHAPTER
from IPython.display import Video
Video(f'build/chapters/{CHAPTER}.mp4', embed=True, width=720)

In [ ]:
# ═══ セル7: 成果物を Drive へ保存 ═══
# Drive: documentary_gpu_output/  （buildフォルダID 1L88uF0w33tJXZTMWM4V_yDzHqmHZfCSb）
import shutil, glob, os
OUT = '/content/drive/MyDrive/documentary_gpu_output'
os.makedirs(OUT, exist_ok=True)
for f in glob.glob('build/chapters/*.mp4') + glob.glob('build/clips/*.mp4'):
    shutil.copy(f, OUT); print('保存:', os.path.basename(f))
print('→', OUT)